# 🚂 Railway Track Defect Detection — Evaluation Results
## Mask2Former-Based Panoptic Segmentation System

This notebook presents the evaluation results of the proposed **Mask2Former-based railway defect detection system**, including:
- Quantitative metrics (mIoU, F1-Score, Precision, Recall, Panoptic Quality)
- Model comparison against baselines (FCN, DeepLabV3+, Mask2Former)
- Per-class performance breakdown
- Training convergence curves
- Instance segmentation analysis
- Confusion matrix
- Failure case analysis (small defects, poor lighting)

> **Note:** All results are simulated to replicate experimental findings from the paper.


In [ ]:
# ─── Imports ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.ndimage import gaussian_filter
import warnings
warnings.filterwarnings('ignore')

# ─── Global Style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':     'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid':       True,
    'grid.alpha':      0.3,
    'grid.linestyle':  '--',
    'figure.dpi':      140,
    'axes.titlesize':  13,
    'axes.labelsize':  11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

# Palette
COLORS = {
    'FCN':        '#6C8EBF',
    'DeepLabV3+': '#F4A261',
    'Mask2Former':'#2A9D8F',
    'accent':     '#E76F51',
    'bg':         '#F8F9FA',
}

np.random.seed(42)
print('✅ Setup complete')

---
## 1. Headline Metrics — Model Comparison

In [ ]:
# ─── Simulated benchmark data ──────────────────────────────────────────────
models = ['FCN', 'DeepLabV3+', 'Mask2Former']

metrics = {
    'mIoU':      [0.55, 0.62, 0.81],   # Mask2Former +30.6% over DeepLabV3+
    'F1-Score':  [0.60, 0.68, 0.86],   # Mask2Former +26.5% over DeepLabV3+
    'Precision': [0.62, 0.70, 0.87],
    'Recall':    [0.58, 0.66, 0.85],
    'PQ':        [None, None, 0.79],    # Panoptic Quality (instance-aware, N/A for semantic models)
}

df = pd.DataFrame(metrics, index=models)
print(df.to_string())

# ─── Grouped bar chart ─────────────────────────────────────────────────────
metric_names = ['mIoU', 'F1-Score', 'Precision', 'Recall']
x = np.arange(len(metric_names))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(COLORS['bg'])
ax.set_facecolor(COLORS['bg'])

for i, (model, color) in enumerate(COLORS.items()):
    if model in ['accent', 'bg']:
        continue
    idx = models.index(model)
    vals = [metrics[m][idx] for m in metric_names]
    bars = ax.bar(x + (i - 1) * width, vals, width, label=model,
                  color=color, edgecolor='white', linewidth=0.8, zorder=3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_ylim(0, 1.08)
ax.set_xticks(x)
ax.set_xticklabels(metric_names, fontsize=10)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Performance Comparison — Railway Defect Detection', fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10, loc='upper left')

# Improvement annotations
for i, m in enumerate(metric_names):
    base = metrics[m][1]  # DeepLabV3+
    proposed = metrics[m][2]  # Mask2Former
    improvement = (proposed - base) / base * 100
    ax.annotate(f'+{improvement:.1f}%', xy=(x[i] + width, proposed + 0.03),
                fontsize=7.5, color=COLORS['accent'], fontweight='bold',
                ha='center')

plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print("\n📌 Mask2Former achieves mIoU=0.81 (+30.6% over DeepLabV3+) and F1=0.86 (+26.5%)")

---
## 2. Per-Class IoU Breakdown

In [ ]:
# ─── Per-class IoU for each model ──────────────────────────────────────────
classes = ['Background', 'Longitudinal\nCrack', 'Transverse\nCrack',
           'Alligator\nCrack', 'Rail\nBreak', 'Surface\nCorrosion']

per_class_iou = {
    'FCN':        [0.92, 0.48, 0.45, 0.41, 0.38, 0.52],
    'DeepLabV3+': [0.94, 0.58, 0.56, 0.52, 0.46, 0.60],
    'Mask2Former':[0.96, 0.82, 0.80, 0.78, 0.74, 0.84],
}

x = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(COLORS['bg'])
ax.set_facecolor(COLORS['bg'])

for i, (model, color) in enumerate([(m, COLORS[m]) for m in models]):
    vals = per_class_iou[model]
    bars = ax.bar(x + (i - 1) * width, vals, width, label=model,
                  color=color, edgecolor='white', linewidth=0.8, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(classes, fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_ylabel('IoU Score', fontsize=11)
ax.set_title('Per-Class IoU — All Models', fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10)

# Mean line for Mask2Former
mean_m2f = np.mean(per_class_iou['Mask2Former'])
ax.axhline(mean_m2f, color=COLORS['Mask2Former'], linestyle='--', linewidth=1.5,
           label=f'Mask2Former mean = {mean_m2f:.2f}')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('per_class_iou.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 3. Training & Validation Convergence Curves

In [ ]:
# ─── Simulated training curves ─────────────────────────────────────────────
epochs = np.arange(1, 51)

def smooth_curve(start, end, epochs, noise=0.015):
    """Simulate a realistic learning curve with exponential convergence."""
    base = end - (end - start) * np.exp(-0.12 * epochs)
    return np.clip(base + np.random.normal(0, noise, len(epochs)), 0, 1)

train_loss_m2f = 1.8 * np.exp(-0.09 * epochs) + 0.18 + np.random.normal(0, 0.02, len(epochs))
val_loss_m2f   = 1.8 * np.exp(-0.085 * epochs) + 0.22 + np.random.normal(0, 0.025, len(epochs))

train_miou_m2f = smooth_curve(0.30, 0.82, epochs, noise=0.012)
val_miou_m2f   = smooth_curve(0.25, 0.78, epochs, noise=0.018)

train_miou_dl  = smooth_curve(0.28, 0.64, epochs, noise=0.012)
val_miou_dl    = smooth_curve(0.23, 0.60, epochs, noise=0.018)

train_miou_fcn = smooth_curve(0.25, 0.57, epochs, noise=0.012)
val_miou_fcn   = smooth_curve(0.20, 0.53, epochs, noise=0.018)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(COLORS['bg'])

# Loss
ax = axes[0]
ax.set_facecolor(COLORS['bg'])
ax.plot(epochs, train_loss_m2f, color=COLORS['Mask2Former'], lw=2, label='Train Loss')
ax.plot(epochs, val_loss_m2f,   color=COLORS['Mask2Former'], lw=2, linestyle='--', label='Val Loss')
ax.fill_between(epochs, train_loss_m2f, val_loss_m2f, alpha=0.08, color=COLORS['Mask2Former'])
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Mask2Former — Training & Validation Loss', fontweight='bold')
ax.legend()

# mIoU comparison
ax = axes[1]
ax.set_facecolor(COLORS['bg'])
ax.plot(epochs, val_miou_m2f, color=COLORS['Mask2Former'], lw=2.5, label='Mask2Former')
ax.plot(epochs, val_miou_dl,  color=COLORS['DeepLabV3+'],  lw=2,   label='DeepLabV3+')
ax.plot(epochs, val_miou_fcn, color=COLORS['FCN'],         lw=2,   label='FCN')
ax.axhline(0.81, color=COLORS['Mask2Former'], linestyle=':', lw=1.2, alpha=0.7)
ax.text(51.5, 0.81, '0.81', color=COLORS['Mask2Former'], va='center', fontsize=8)
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation mIoU')
ax.set_title('Validation mIoU — Model Convergence Comparison', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('training_curves.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 4. Confusion Matrix — Mask2Former

In [ ]:
# ─── Simulated pixel-level confusion matrix ────────────────────────────────
class_names = ['Background', 'Long. Crack', 'Trans. Crack',
               'Alligator Crack', 'Rail Break', 'Surface Corrosion']

# Build a realistic confusion matrix
cm_raw = np.array([
    [9850,   30,   40,   25,   20,   35],
    [  45,  820,   28,   18,   12,   22],
    [  50,   22,  790,   20,   15,   18],
    [  38,   20,   25,  760,   22,   15],
    [  30,   15,   18,   20,  710,   17],
    [  42,   25,   22,   15,   18,  838],
], dtype=float)

# Normalize by row
cm_norm = cm_raw / cm_raw.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor(COLORS['bg'])

for ax, data, fmt, title in zip(
    axes,
    [cm_raw, cm_norm],
    ['.0f', '.2f'],
    ['Confusion Matrix (Raw Counts)', 'Confusion Matrix (Row-Normalised)']):

    ax.set_facecolor(COLORS['bg'])
    cmap = LinearSegmentedColormap.from_list('teal', ['#EAFAF1', '#2A9D8F'])
    sns.heatmap(data, annot=True, fmt=fmt, cmap=cmap,
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, linecolor='white',
                ax=ax, cbar_kws={'shrink': 0.8})
    ax.set_title(f'Mask2Former — {title}', fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 5. Instance Separation Capability

In [ ]:
# ─── Simulate a synthetic railway track image with overlapping instances ───
H, W = 300, 600
track_bg = np.ones((H, W, 3)) * 0.35  # dark grey ballast

# Add rail lines
track_bg[120:180, :, :] = 0.55
track_bg[130:170, :, :] = 0.60
track_bg[100:115, :, :] = 0.48
track_bg[185:200, :, :] = 0.48

# Rail texture noise
noise = np.random.normal(0, 0.04, (H, W, 3))
img = np.clip(track_bg + noise, 0, 1)
img = gaussian_filter(img, sigma=0.8)

# ─── Masks for 3 instances ─────────────────────────────────────────────────
def make_crack_mask(y0, x0, length, thickness, angle_deg=0):
    mask = np.zeros((H, W), dtype=bool)
    angle = np.deg2rad(angle_deg)
    for t in np.linspace(-length//2, length//2, length*3):
        cx = int(x0 + t * np.cos(angle))
        cy = int(y0 + t * np.sin(angle))
        for dy in range(-thickness, thickness+1):
            for dx in range(-thickness, thickness+1):
                ny, nx = cy+dy, cx+dx
                if 0 <= ny < H and 0 <= nx < W:
                    mask[ny, nx] = True
    return mask

instance_masks = [
    make_crack_mask(150, 150, 80, 4, 85),   # Longitudinal crack
    make_crack_mask(145, 310, 70, 3, 5),    # Transverse crack
    make_crack_mask(155, 460, 90, 4, 80),   # Another longitudinal
]
instance_colors = [(0.9, 0.3, 0.3), (0.3, 0.9, 0.5), (0.3, 0.5, 0.95)]
instance_labels = ['Instance #1\nLong. Crack', 'Instance #2\nTrans. Crack', 'Instance #3\nLong. Crack']

# ─── Semantic (flat) mask
semantic_mask = np.zeros((H, W, 3))
for m in instance_masks:
    semantic_mask[m] = [0.9, 0.4, 0.1]  # single class colour

# ─── Instance-coloured overlay
instance_overlay = img.copy()
for m, c in zip(instance_masks, instance_colors):
    for ch, val in enumerate(c):
        instance_overlay[:, :, ch][m] = val

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor(COLORS['bg'])
titles = ['Original Track Image', 'Semantic Mask (DeepLabV3+)\n— All cracks = single class', 'Instance Mask (Mask2Former)\n— Unique ID per defect']
displays = [img, np.clip(img * 0.5 + semantic_mask * 0.8, 0, 1), instance_overlay]

for ax, disp, title in zip(axes, displays, titles):
    ax.imshow(disp)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.axis('off')

# Legend on instance panel
patches = [mpatches.Patch(color=c, label=l) for c, l in zip(instance_colors, instance_labels)]
axes[2].legend(handles=patches, loc='lower right', fontsize=8, framealpha=0.85)

fig.suptitle('Instance-Level Separation: Mask2Former vs Semantic Models', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('instance_separation.png', bbox_inches='tight', dpi=150)
plt.show()
print("\n🔑 Mask2Former assigns unique IDs to each crack — impossible for FCN / DeepLabV3+")

---
## 6. Masked Attention Effectiveness — Background vs Defect Attention Maps

In [ ]:
# ─── Simulated attention maps ──────────────────────────────────────────────
H2, W2 = 200, 400

def make_attention(focus_spots, spread, H=H2, W=W2, bg_noise=0.1):
    attn = np.random.uniform(0, bg_noise, (H, W))
    for (y, x), s in zip(focus_spots, spread):
        yy, xx = np.mgrid[0:H, 0:W]
        attn += np.exp(-((yy - y)**2 + (xx - x)**2) / (2 * s**2))
    return np.clip(attn / attn.max(), 0, 1)

# CNN attention — diffuse, attracted to ballast noise
cnn_attn = make_attention(
    [(60, 80), (90, 200), (70, 320), (110, 150), (80, 350), (100, 50)],
    [35, 40, 30, 45, 25, 38], bg_noise=0.25)

# Mask2Former masked attention — tight on defect region
m2f_attn = make_attention(
    [(100, 120), (100, 260), (100, 380)],
    [12, 10, 14], bg_noise=0.03)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor(COLORS['bg'])
cmap_hot = plt.cm.inferno

for ax, attn, title, desc in zip(
    axes,
    [cnn_attn, m2f_attn],
    ['CNN Cross-Attention (DeepLabV3+)', 'Masked Cross-Attention (Mask2Former)'],
    ['Diffuse — distracted by ballast, shadows & vegetation',
     'Constrained to within-mask regions — focused on defect pixels only']):

    im = ax.imshow(attn, cmap=cmap_hot, vmin=0, vmax=1)
    ax.set_title(f'{title}\n{desc}', fontsize=9, fontweight='bold')
    ax.axis('off')
    fig.colorbar(im, ax=ax, shrink=0.8, label='Attention weight')

fig.suptitle('Attention Heatmaps — Masked vs Unmasked Cross-Attention', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('attention_maps.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 7. Panoptic Quality Breakdown

In [ ]:
# ─── Panoptic Quality = Segmentation Quality × Recognition Quality ──────────
pq_data = {
    'Class':              class_names,
    'PQ':  [0.92, 0.80, 0.78, 0.76, 0.72, 0.82],
    'SQ':  [0.96, 0.85, 0.83, 0.81, 0.78, 0.87],   # Segmentation Quality
    'RQ':  [0.96, 0.94, 0.94, 0.94, 0.92, 0.94],   # Recognition Quality
}
pq_df = pd.DataFrame(pq_data).set_index('Class')
print(pq_df.round(3).to_string(), '\n')
print(f"Mean PQ  = {pq_df['PQ'].mean():.3f}")
print(f"Mean SQ  = {pq_df['SQ'].mean():.3f}")
print(f"Mean RQ  = {pq_df['RQ'].mean():.3f}")

x = np.arange(len(class_names))
width = 0.27

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(COLORS['bg'])
ax.set_facecolor(COLORS['bg'])

b1 = ax.bar(x - width, pq_df['PQ'], width, label='PQ (Panoptic Quality)', color='#2A9D8F', edgecolor='white')
b2 = ax.bar(x,         pq_df['SQ'], width, label='SQ (Segmentation Quality)', color='#57CC99', edgecolor='white')
b3 = ax.bar(x + width, pq_df['RQ'], width, label='RQ (Recognition Quality)', color='#F4A261', edgecolor='white')

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x); ax.set_xticklabels(class_names, fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Mask2Former — Panoptic Quality (PQ = SQ × RQ) per Class', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('panoptic_quality.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 8. Failure Case Analysis — Small Defects & Poor Lighting

In [ ]:
# ─── IoU degradation as defect size decreases ──────────────────────────────
defect_sizes   = [5, 20, 50, 100, 200, 400, 700, 1000]  # pixels
iou_mask2former = [0.31, 0.51, 0.68, 0.79, 0.83, 0.85, 0.85, 0.86]
iou_deeplab     = [0.18, 0.32, 0.50, 0.62, 0.65, 0.66, 0.67, 0.68]
iou_fcn         = [0.12, 0.25, 0.42, 0.53, 0.56, 0.57, 0.57, 0.58]

# ─── IoU degradation under poor lighting ───────────────────────────────────
brightness_levels = [10, 20, 30, 50, 70, 100]   # % of normal brightness
iou_m2f_light     = [0.42, 0.58, 0.69, 0.78, 0.82, 0.85]
iou_dl_light      = [0.28, 0.40, 0.52, 0.63, 0.67, 0.69]
iou_fcn_light     = [0.20, 0.30, 0.42, 0.54, 0.58, 0.60]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(COLORS['bg'])

# ── Left: defect size
ax = axes[0]
ax.set_facecolor(COLORS['bg'])
ax.plot(defect_sizes, iou_mask2former, 'o-', color=COLORS['Mask2Former'], lw=2.5, ms=6, label='Mask2Former')
ax.plot(defect_sizes, iou_deeplab,     's-', color=COLORS['DeepLabV3+'],  lw=2,   ms=5, label='DeepLabV3+')
ax.plot(defect_sizes, iou_fcn,         '^-', color=COLORS['FCN'],         lw=2,   ms=5, label='FCN')
ax.axvline(50, color=COLORS['accent'], linestyle='--', lw=1.5, label='50-px threshold')
ax.fill_betweenx([0, 1], 0, 50, alpha=0.07, color=COLORS['accent'])
ax.text(5, 0.92, '⚠ Failure zone\n(< 50 px)', color=COLORS['accent'], fontsize=8)
ax.set_xscale('log')
ax.set_xlabel('Defect Size (pixels, log scale)')
ax.set_ylabel('IoU')
ax.set_title('Performance vs Defect Size', fontweight='bold')
ax.legend(fontsize=9)

# ── Right: lighting
ax = axes[1]
ax.set_facecolor(COLORS['bg'])
ax.plot(brightness_levels, iou_m2f_light, 'o-', color=COLORS['Mask2Former'], lw=2.5, ms=6, label='Mask2Former')
ax.plot(brightness_levels, iou_dl_light,  's-', color=COLORS['DeepLabV3+'],  lw=2,   ms=5, label='DeepLabV3+')
ax.plot(brightness_levels, iou_fcn_light, '^-', color=COLORS['FCN'],         lw=2,   ms=5, label='FCN')
ax.fill_betweenx([0, 1], 0, 30, alpha=0.07, color=COLORS['accent'])
ax.text(11, 0.92, '⚠ Poor\nlighting', color=COLORS['accent'], fontsize=8)
ax.set_xlabel('Image Brightness (% of normal)')
ax.set_ylabel('IoU')
ax.set_title('Performance vs Lighting Conditions', fontweight='bold')
ax.legend(fontsize=9)

fig.suptitle('Failure Case Analysis — Edge Conditions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('failure_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print("\n⚠️  Model performance degrades for defects < 50 pixels and brightness < 30% of normal")

---
## 9. Oversized Mask Analysis

In [ ]:
# ─── Oversized mask frequency vs defect size ───────────────────────────────
size_bins   = [0, 25, 50, 75, 100, 150, 200, 300, 500]
labels_bins = ['0-25', '25-50', '50-75', '75-100', '100-150', '150-200', '200-300', '300-500']
oversized_pct = [48.0, 31.0, 18.0, 10.0, 5.5, 3.0, 1.8, 0.9]  # % images with oversized masks
sample_counts = [42, 68, 95, 120, 98, 85, 72, 55]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(COLORS['bg'])

# Bar — oversized %
ax = axes[0]
ax.set_facecolor(COLORS['bg'])
bar_colors = [COLORS['accent'] if p > 15 else COLORS['Mask2Former'] for p in oversized_pct]
bars = ax.bar(labels_bins, oversized_pct, color=bar_colors, edgecolor='white', lw=0.8)
for bar, pct in zip(bars, oversized_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_xlabel('Defect Size (pixels)')
ax.set_ylabel('Oversized Mask Frequency (%)')
ax.set_title('Oversized Mask Frequency\nvs Defect Size', fontweight='bold')
ax.tick_params(axis='x', rotation=25)
red_patch = mpatches.Patch(color=COLORS['accent'], label='High-error zone (>15%)')
green_patch = mpatches.Patch(color=COLORS['Mask2Former'], label='Acceptable (<15%)')
ax.legend(handles=[red_patch, green_patch], fontsize=9)

# Scatter — sample counts
ax = axes[1]
ax.set_facecolor(COLORS['bg'])
sc = ax.scatter(sample_counts, oversized_pct,
                c=oversized_pct, cmap='RdYlGn_r', s=120, zorder=3, edgecolors='white', lw=0.8)
for cnt, pct, lbl in zip(sample_counts, oversized_pct, labels_bins):
    ax.annotate(lbl, (cnt, pct), textcoords='offset points', xytext=(5, 4), fontsize=7.5)
plt.colorbar(sc, ax=ax, label='Oversized %')
ax.set_xlabel('Number of Test Samples')
ax.set_ylabel('Oversized Mask Frequency (%)')
ax.set_title('Sample Count vs Oversized Mask Rate', fontweight='bold')

plt.tight_layout()
plt.savefig('oversized_mask_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 10. Summary Dashboard

In [ ]:
# ─── Full summary dashboard ────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#1A1A2E')
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.4)

title_kw  = dict(fontsize=10, fontweight='bold', color='white', pad=8)
label_kw  = dict(fontsize=8,  color='#AAAAAA')
tick_kw   = dict(colors='#AAAAAA')
bg_axes   = '#16213E'
spine_col = '#0F3460'

def style_ax(ax):
    ax.set_facecolor(bg_axes)
    for sp in ax.spines.values():
        sp.set_color(spine_col)
    ax.tick_params(colors='#AAAAAA')
    ax.grid(True, alpha=0.15, color='white', linestyle='--')
    ax.yaxis.label.set_color('#AAAAAA')
    ax.xaxis.label.set_color('#AAAAAA')

# ── Radar / Spider chart ───────────────────────────────────────────────────
ax_radar = fig.add_subplot(gs[0:2, 0:2], polar=True)
ax_radar.set_facecolor(bg_axes)
radar_metrics = ['mIoU', 'F1-Score', 'Precision', 'Recall', 'PQ (norm.)']
m2f_vals   = [0.81, 0.86, 0.87, 0.85, 0.79]
dl_vals    = [0.62, 0.68, 0.70, 0.66, 0.00]
fcn_vals   = [0.55, 0.60, 0.62, 0.58, 0.00]
N = len(radar_metrics)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

for vals, model, color in [
    (m2f_vals, 'Mask2Former', '#2A9D8F'),
    (dl_vals,  'DeepLabV3+',  '#F4A261'),
    (fcn_vals, 'FCN',         '#6C8EBF')]:
    v = vals + vals[:1]
    ax_radar.plot(angles, v, color=color, lw=2, label=model)
    ax_radar.fill(angles, v, color=color, alpha=0.12)

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(radar_metrics, color='white', fontsize=9)
ax_radar.set_ylim(0, 1)
ax_radar.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax_radar.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], color='#888', fontsize=7)
ax_radar.set_facecolor('#16213E')
ax_radar.spines['polar'].set_color('#0F3460')
ax_radar.grid(color='#333366', alpha=0.5)
ax_radar.set_title('Overall Metric Comparison (Radar)', color='white', fontsize=11, fontweight='bold', pad=20)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8,
                facecolor='#1A1A2E', edgecolor='#333366', labelcolor='white')

# ── Learning curve ─────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2:4])
style_ax(ax2)
ax2.plot(epochs, val_miou_m2f, color='#2A9D8F', lw=2, label='Mask2Former')
ax2.plot(epochs, val_miou_dl,  color='#F4A261', lw=1.8, label='DeepLabV3+')
ax2.plot(epochs, val_miou_fcn, color='#6C8EBF', lw=1.8, label='FCN')
ax2.set_title('Val mIoU — Convergence', **title_kw)
ax2.legend(fontsize=8, facecolor=bg_axes, edgecolor=spine_col, labelcolor='white')
ax2.set_xlabel('Epoch', **label_kw)
ax2.set_ylabel('Val mIoU', **label_kw)

# ── Per-class bar ──────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 2:4])
style_ax(ax3)
short_cls = ['BG', 'Long.\nCrack', 'Trans.\nCrack', 'Allig.\nCrack', 'Rail\nBreak', 'Surf.\nCorr.']
bars3 = ax3.bar(short_cls, per_class_iou['Mask2Former'],
                color=['#2A9D8F','#57CC99','#48CAE4','#0077B6','#F4A261','#E76F51'],
                edgecolor='#1A1A2E', lw=0.6)
for bar, val in zip(bars3, per_class_iou['Mask2Former']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.2f}', ha='center', va='bottom', fontsize=7.5, color='white')
ax3.set_ylim(0, 1.1)
ax3.set_title('Mask2Former — Per-Class IoU', **title_kw)
ax3.set_ylabel('IoU', **label_kw)

# ── Metric summary table ───────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, :])
ax4.set_facecolor('#16213E')
ax4.axis('off')

summary_data = [
    ['Model',        'mIoU',  'F1-Score', 'Precision', 'Recall', 'PQ',   'Instance Sep.'],
    ['FCN',          '0.55',  '0.60',     '0.62',      '0.58',   'N/A',  '✗'],
    ['DeepLabV3+',   '0.62',  '0.68',     '0.70',      '0.66',   'N/A',  '✗'],
    ['Mask2Former',  '0.81',  '0.86',     '0.87',      '0.85',   '0.79', '✓'],
    ['Improvement',  '+30.6%','+26.5%',   '+24.3%',    '+28.8%', '—',    '—'],
]

row_colors  = ['#0F3460'] + ['#16213E'] * 3 + ['#1A2E1A']
cell_colors = [[rc]*7 for rc in row_colors]
t = ax4.table(cellText=summary_data[1:], colLabels=summary_data[0],
              cellLoc='center', loc='center', cellColours=cell_colors[1:])
t.auto_set_font_size(False)
t.set_fontsize(9)
t.scale(1, 2.2)
for key, cell in t.get_celld().items():
    cell.set_edgecolor('#0F3460')
    cell.set_text_props(color='white')
    if key[0] == 0:  # header
        cell.set_facecolor('#0F3460')
        cell.set_text_props(fontweight='bold', color='#57CC99')
    if key[0] == 4:  # improvement row
        cell.set_facecolor('#1A2E1A')
        cell.set_text_props(color='#57CC99', fontweight='bold')

ax4.set_title('Summary Table — All Models', color='white', fontsize=11, fontweight='bold', pad=10)

fig.suptitle('🚂  Railway Track Defect Detection — Evaluation Dashboard',
             fontsize=16, fontweight='bold', color='white', y=0.98)

plt.savefig('evaluation_dashboard.png', bbox_inches='tight', dpi=150, facecolor='#1A1A2E')
plt.show()
print('\n✅ Dashboard saved to evaluation_dashboard.png')

---
## 11. Key Findings Summary

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║     MASK2FORMER RAILWAY DEFECT DETECTION — KEY RESULTS SUMMARY          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  📊 OVERALL PERFORMANCE                                                  ║
║     mIoU           : 0.81  (+30.6% vs DeepLabV3+)                        ║
║     F1-Score       : 0.86  (+26.5% vs DeepLabV3+)                        ║
║     Precision      : 0.87                                                 ║
║     Recall         : 0.85                                                 ║
║     Panoptic Quality (PQ) : 0.79 (instance-aware)                        ║
║                                                                          ║
║  🔍 KEY ADVANTAGES                                                       ║
║     ✓ Instance-level separation (unique IDs per defect)                  ║
║     ✓ Masked cross-attention avoids background distraction               ║
║     ✓ Superior on small-to-medium defect regions                         ║
║                                                                          ║
║  ⚠  LIMITATIONS                                                         ║
║     ✗ Oversized masks for defects < 50 pixels (~48% error rate)          ║
║     ✗ Performance degrades under poor lighting (< 30% brightness)        ║
║                                                                          ║
║  🔭 FUTURE WORK                                                          ║
║     → Larger annotated datasets                                           ║
║     → Multi-scale attention mechanisms                                    ║
║     → Low-light image enhancement preprocessing                          ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")